# Compile and run your own MapReduce program (Google Colab)

In Session 1 you *used* the programs that come with Hadoop. Today you **write, build and run your own**: a Java project built with **Maven**,
run first in *local* mode and then on your pseudo-cluster, selecting the mode with Hadoop's **Configuration API** (`-conf`, `-D`).

| Part | What you will do |
|---|---|
| 0 | 🔁 Rebuild your Hadoop pseudo-cluster (two ready-made cells) |
| 1 | Install **Maven** |
| 2 | Download and explore the project (Mapper, Reducer, Driver) |
| 3 | Build it: `mvn compile`, `mvn package` |
| 4 | Run it **locally** and **on the cluster** with the Configuration API |
| 5 | 🧩 **Final exercise**: from *maximum* to *average* temperature per year, on **your own** data file |

**How it works.** Same as in Session 1: every step has a *goal* and the *expected result*, the code cells are **empty: you write the commands**
(except Part 0), and every question has a box below it where **you write the answer**. Run the cells **in order**.

> ⚠️ A Colab runtime is temporary: if it is recycled you lose everything, including the cluster. Run Part 0 again and continue.

# Part 0 – Rebuild the cluster (ready-made)

You already know every step of this from Session 1, so this time **the cluster comes pre-built**. Run the two cells below, in order (≈ 3-4 min).
They install Java 17 and Hadoop 3.5.0, configure pseudo-distributed mode, start passwordless SSH, format HDFS and start HDFS + YARN.
At the end `jps` must list **NameNode, DataNode, SecondaryNameNode, ResourceManager and NodeManager**.

In [1]:
import os
HADOOP_VERSION = "3.5.0"
HADOOP_HOME = "/usr/local/hadoop"
JAVA_HOME = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ.update({
    "HADOOP_VERSION": HADOOP_VERSION, "JAVA_HOME": JAVA_HOME, "HADOOP_HOME": HADOOP_HOME,
    "HADOOP_MAPRED_HOME": HADOOP_HOME, "HADOOP_COMMON_HOME": HADOOP_HOME,
    "HADOOP_HDFS_HOME": HADOOP_HOME, "YARN_HOME": HADOOP_HOME,
    "HADOOP_CONF_DIR": HADOOP_HOME + "/etc/hadoop",
})
os.environ["PATH"] = f"{JAVA_HOME}/bin:{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]
print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-17-openjdk-amd64


In [2]:
%%bash
set -e
# --- Java 17 and Hadoop 3.5.0
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless openssh-server > /dev/null
if [ ! -d /usr/local/hadoop ]; then
  cd /content
  F=hadoop-$HADOOP_VERSION/hadoop-$HADOOP_VERSION.tar.gz
  wget -q https://dlcdn.apache.org/hadoop/common/$F || wget -q https://archive.apache.org/dist/hadoop/common/$F
  tar -xzf hadoop-$HADOOP_VERSION.tar.gz
  mv hadoop-$HADOOP_VERSION /usr/local/hadoop
fi
# --- passwordless SSH to localhost (Colab: own sshd on port 22)
mkdir -p /run/sshd
pgrep -f "sshd -p 22" > /dev/null || /usr/sbin/sshd -p 22 -o ListenAddress=127.0.0.1
sleep 2
mkdir -p ~/.ssh && chmod 700 ~/.ssh
[ -f ~/.ssh/id_rsa ] || ssh-keygen -t rsa -P "" -f ~/.ssh/id_rsa -q
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 0600 ~/.ssh/authorized_keys
printf "Host *\n  StrictHostKeyChecking no\n  UserKnownHostsFile /dev/null\n  LogLevel ERROR\n" > ~/.ssh/config
chmod 600 ~/.ssh/config
# --- configuration files
cat >> /usr/local/hadoop/etc/hadoop/hadoop-env.sh <<EOF
export JAVA_HOME=$JAVA_HOME
export HADOOP_HOME=$HADOOP_HOME
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export YARN_RESOURCEMANAGER_USER=root
export YARN_NODEMANAGER_USER=root
EOF
cat > /usr/local/hadoop/etc/hadoop/core-site.xml <<'EOF'
<?xml version="1.0"?>
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
</configuration>
EOF
cat > /usr/local/hadoop/etc/hadoop/hdfs-site.xml <<'EOF'
<?xml version="1.0"?>
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.secondary.http-address</name><value>localhost:9868</value></property>
</configuration>
EOF
cat > /usr/local/hadoop/etc/hadoop/mapred-site.xml <<'EOF'
<?xml version="1.0"?>
<configuration>
  <property><name>mapreduce.framework.name</name><value>yarn</value></property>
  <property><name>yarn.app.mapreduce.am.env</name><value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value></property>
  <property><name>mapreduce.map.env</name><value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value></property>
  <property><name>mapreduce.reduce.env</name><value>HADOOP_MAPRED_HOME=/usr/local/hadoop</value></property>
</configuration>
EOF
cat > /usr/local/hadoop/etc/hadoop/yarn-site.xml <<'EOF'
<?xml version="1.0"?>
<configuration>
  <property><name>yarn.nodemanager.aux-services</name><value>mapreduce_shuffle</value></property>
  <property><name>yarn.nodemanager.aux-services.mapreduce.shuffle.class</name><value>org.apache.hadoop.mapred.ShuffleHandler</value></property>
</configuration>
EOF
# --- format HDFS and start the daemons
hdfs namenode -format -force > /dev/null 2>&1
start-dfs.sh
start-yarn.sh
sleep 15
jps

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [localhost]
Starting resourcemanager
Starting nodemanagers
4752 ResourceManager
4388 SecondaryNameNode
5300 Jps
4220 DataNode
4861 NodeManager
4109 NameNode


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


# Part 1 – Maven

**Maven** (a Yiddish word meaning *accumulator of knowledge*) builds and manages Java projects. It reads a **POM** (*Project Object Model*, the `pom.xml` file) that lists
the project's **dependencies** (it downloads them for you, here the Hadoop client libraries) and the **plugins** that compile and package the code.
Once you know one Maven project, you know how all of them build: the same commands everywhere.

## 1.1 Install Maven

Install Maven with `apt` and check the installation with `mvn -version`.

<details><summary>💡 Hint</summary>

The package is called `maven`. Use `-y`. `mvn -version` prints the Maven version and also the Java it will use.

</details>

✅ **Expected:** `Apache Maven 3.x` and `Java version: 17.x` (from `JAVA_HOME`).

In [3]:
%%bash
apt-get -qq install -y maven > /dev/null
mvn -version

Apache Maven 3.8.7
Maven home: /usr/share/maven
Java version: 17.0.21.1, vendor: Ubuntu, runtime: /usr/lib/jvm/java-17-openjdk-amd64
Default locale: en_US, platform encoding: UTF-8
OS name: "linux", version: "6.6.159+", arch: "amd64", family: "unix"


❓ **Question:** Which Java version does Maven report? Why is it the one we want and not Colab's default Java 21? (Look at the variable that we set in Part 0.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

# Part 2 – Download and explore the project

## 2.1 Clone the project

The sample project is on GitHub: `https://github.com/AntSanVar/DistributedSystem`. Clone it into `/content` (just the last commit is enough) and list its files **as a list** (the `find` command, hiding the hidden `.git` folder).

<details><summary>💡 Hint</summary>

`git clone --depth 1 <url>`. Then `cd` into the folder; `find . -not -path './.git*' | sort`.

</details>

✅ **Expected:** `pom.xml`, `run.sh`, `conf/`, `input/sample.txt` and four `.java` files in `src/main/java`.

In [4]:
%%bash
cd /content
rm -rf DistributedSystem
git clone -q --depth 1 https://github.com/AntSanVar/DistributedSystem.git
cd DistributedSystem
find . -not -path './.git*' | sort

.
./conf
./conf/hadoop-cluster.template.xml
./conf/hadoop-localhost.xml
./conf/hadoop-local.xml
./input
./input/sample.txt
./pom.xml
./README.md
./run.sh
./src
./src/main
./src/main/java
./src/main/java/MaxTemperatureDriver.java
./src/main/java/MaxTemperatureMapper.java
./src/main/java/MaxTemperatureReducer.java
./src/main/java/NcdcRecordParser.java


❓ **Question:** Which files are Java sources? Which one is the POM? Which folder holds the configuration files, and which one the sample input?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 2.2 The input data

The project processes weather records in the **NCDC** format (National Climatic Data Center): one fixed-width line per measurement. Print `input/sample.txt`.

The **year** is in characters 16-19, the **air temperature** (tenths of °C, with sign) in characters 88-92 and a **quality code** in character 93.

<details><summary>💡 Hint</summary>

`cat` the file (with the full path `/content/DistributedSystem/input/sample.txt`).

</details>

✅ **Expected:** Five long lines for the years 1950 and 1949.

In [5]:
!cat /content/DistributedSystem/input/sample.txt

0067011990999991950051507004+68750+023550FM-12+038299999V0203301N00671220001CN9999999N9+00001+99999999999
0043011990999991950051512004+68750+023550FM-12+038299999V0203201N00671220001CN9999999N9+00221+99999999999
0043011990999991950051518004+68750+023550FM-12+038299999V0203201N00261220001CN9999999N9-00111+99999999999
0043012650999991949032412004+62300+010750FM-12+048599999V0202701N00461220001CN0500001N9+01111+99999999999
0043012650999991949032418004+62300+010750FM-12+048599999V0202701N00461220001CN0500001N9+00781+99999999999

❓ **Question:** Looking at the 5 lines: which years appear? What temperature does the first line have (remember it is in tenths of a degree)?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 2.3 The Mapper

Read `MaxTemperatureMapper.java` and `NcdcRecordParser.java`.

<details><summary>💡 Hint</summary>

`cat` both files inside `src/main/java`.

</details>

✅ **Expected:** Input key = byte offset (`LongWritable`), input value = the line (`Text`); output key = year (`Text`), output value = temperature (`IntWritable`). Nothing is written when the reading is missing (`9999`) or the quality code is not in `[01459]`.

In [6]:
%%bash
cd /content/DistributedSystem/src/main/java
cat MaxTemperatureMapper.java NcdcRecordParser.java

// MaxTemperatureMapper A Mapper that uses a utility class to parse records

import java.io.IOException;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Mapper;

public class MaxTemperatureMapper
    extends Mapper<LongWritable, Text, Text, IntWritable> {

  private NcdcRecordParser parser = new NcdcRecordParser();

  @Override
  public void map(LongWritable key, Text value, Context context)
      throws IOException, InterruptedException {

    parser.parse(value);
    if (parser.isValidTemperature()) {
      context.write(new Text(parser.getYear()),
          new IntWritable(parser.getAirTemperature()));
    }
  }
}

//NcdcRecordParser A class for parsing weather records in NCDC format
import org.apache.hadoop.io.Text;

public class NcdcRecordParser {
  
  private static final int MISSING_TEMPERATURE = 9999;
  
  private String year;
  private int airTemperature;
  private String q

❓ **Question:** What are the 4 type parameters of `Mapper<LongWritable, Text, Text, IntWritable>`? What does `map()` write to the context, and when does it write nothing?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 2.4 The Reducer and the Driver

Read `MaxTemperatureReducer.java` and `MaxTemperatureDriver.java`.

<details><summary>💡 Hint</summary>

`cat` the two files. A Combiner is a *mini-reducer* that runs on the map side before the shuffle.

</details>

✅ **Expected:** The Reducer receives (year, all its temperatures) and writes (year, maximum). `job.setCombinerClass(MaxTemperatureReducer.class)`; the maximum of maxima is the maximum, so applying it twice gives the same result.

In [7]:
%%bash
cd /content/DistributedSystem/src/main/java
cat MaxTemperatureReducer.java MaxTemperatureDriver.java

//MaxTemperatureReducer Reducer for maximum temperature example
import java.io.IOException;

import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;

public class MaxTemperatureReducer
  extends Reducer<Text, IntWritable, Text, IntWritable> {

  @Override
  public void reduce(Text key, Iterable<IntWritable> values,
      Context context)
      throws IOException, InterruptedException {
    
    int maxValue = Integer.MIN_VALUE;
    for (IntWritable value : values) {
      maxValue = Math.max(maxValue, value.get());
    }
    context.write(key, new IntWritable(maxValue));
  }
}

//MaxTemperatureDriver Application to find the maximum temperature
import org.apache.hadoop.conf.Configured;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapredu

❓ **Question:** What does the Reducer receive and what does it write? Which line of the Driver registers the Combiner, and why can the same class be used as Reducer **and** Combiner here?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 2.5 The POM

Open `pom.xml` and find: the **Hadoop version** (a *property*), the **main class** of the jar and the **final name** of the jar.

<details><summary>💡 Hint</summary>

`grep -n -E 'hadoop.version|mainClass|fileName' pom.xml`. In Maven a property can be overridden on the command line with `-Dname=value`.

</details>

✅ **Expected:** `hadoop.version` is 3.4.2 (older than our 3.5.0); `-Dhadoop.version=3.5.0` overrides it.

In [8]:
%%bash
cd /content/DistributedSystem
grep -n -E "hadoop.version|mainClass|fileName|<source>" pom.xml

10:		<hadoop.version>3.5.0</hadoop.version>
11:		<hadoop.mainClass>MaxTemperatureDriver</hadoop.mainClass>
12:		<hadoop.fileName>hadoop-practice</hadoop.fileName>
19:			<version>${hadoop.version}</version>
23:		<finalName>${hadoop.fileName}</finalName>
30:					<source>17</source>
41:							<mainClass>${hadoop.mainClass}</mainClass>


❓ **Question:** Which Hadoop version does the POM ask for? It is not the one we installed: how would you change it **without editing the file**? (You will do it in the next part.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

# Part 3 – Build with Maven

## 3.1 Compile

Compile the project against **our** Hadoop version (override the property of the POM). The first time Maven downloads all the dependencies (1-2 min); `-q` hides most messages.

Then list the folder `target/classes`.

<details><summary>💡 Hint</summary>

`mvn -q -Dhadoop.version=3.5.0 compile` inside the project folder (or use the `$HADOOP_VERSION` variable). Warnings about *deprecated API* are normal.

</details>

✅ **Expected:** The folder contains `MaxTemperatureMapper.class`, `MaxTemperatureReducer.class`, `MaxTemperatureDriver.class` and `NcdcRecordParser.class`.

In [9]:
%%bash
cd /content/DistributedSystem
mvn -q -Dhadoop.version=$HADOOP_VERSION compile
ls target/classes

MaxTemperatureDriver.class
MaxTemperatureMapper.class
MaxTemperatureReducer.class
NcdcRecordParser.class


## 3.2 Package the JAR

Package the classes in a **JAR** (`package` goal, skipping tests). Then check that `hadoop-practice.jar` exists in the project folder and list its content (`jar tf`).

<details><summary>💡 Hint</summary>

Same options as before, goal `package`. The POM writes the jar to the project root (`outputDirectory`).

</details>

✅ **Expected:** `hadoop-practice.jar` (a few KB) containing the 4 classes and `META-INF/MANIFEST.MF`.

In [10]:
%%bash
cd /content/DistributedSystem
mvn -q -Dhadoop.version=$HADOOP_VERSION package -DskipTests
ls -l hadoop-practice.jar
jar tf hadoop-practice.jar

-rw-r--r-- 1 root root 6133 Nov 23 11:46 hadoop-practice.jar
META-INF/
META-INF/MANIFEST.MF
NcdcRecordParser.class
MaxTemperatureMapper.class
MaxTemperatureReducer.class
MaxTemperatureDriver.class
META-INF/maven/
META-INF/maven/com.hadoopbook/
META-INF/maven/com.hadoopbook/Hadoop-Practice/
META-INF/maven/com.hadoopbook/Hadoop-Practice/pom.xml
META-INF/maven/com.hadoopbook/Hadoop-Practice/pom.properties


❓ **Question:** What does `mvn package` do that `mvn compile` does not? Where does Maven put the jar? (look at the POM)

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

# Part 4 – Run it: the Configuration API

Hadoop reads its configuration (which file system? which framework?) from XML files. Our Driver extends `Configured` and is launched with `ToolRunner`, so it **understands the generic options**
`-conf <file>` (load an extra configuration file) and `-D property=value` (override one property) *before* its own arguments.
That is the **Configuration API**: the same jar runs locally or on the cluster just by changing the configuration.

## 4.1 Look at the two configurations

The project has two configuration files in `conf/`. Print `hadoop-local.xml` and `hadoop-localhost.xml`.

<details><summary>💡 Hint</summary>

`cat conf/*.xml`.

</details>

✅ **Expected:** local → `file:///` + `local`; localhost → `hdfs://localhost:9000/` + `yarn` (and the ResourceManager address).

In [11]:
%%bash
cd /content/DistributedSystem
cat conf/hadoop-local.xml conf/hadoop-localhost.xml

<?xml version="1.0"?>
<configuration>

  <property>
    <name>fs.defaultFS</name>
    <value>file:///</value>
  </property>
  
  <property>
    <name>mapreduce.framework.name</name>
    <value>local</value>
  </property>
  
</configuration><?xml version="1.0"?>
<configuration>

  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000/</value>
  </property>

  <property>
    <name>mapreduce.framework.name</name>
    <value>yarn</value>
  </property>

  <property>
    <name>yarn.resourcemanager.address</name>
    <value>localhost:8032</value>
  </property>
  
</configuration>


❓ **Question:** What are the values of `fs.defaultFS` and `mapreduce.framework.name` in each one? Which one means 'standalone' and which one 'my pseudo-cluster'?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 4.2 Run locally

Run the jar in **local mode** over `input/` (a local folder), writing to `output_local`. Use `hadoop jar <jar> -conf conf/hadoop-local.xml <input> <output>`. Remove the output folder first (a job refuses to overwrite it) and print the result.

<details><summary>💡 Hint</summary>

`rm -rf output_local` first. The result is in `output_local/part-r-00000`; `cat output_local/*`.

</details>

✅ **Expected:** `1949	111` and `1950	22` (tenths of °C: 11.1 °C and 2.2 °C). It does not appear in YARN: local mode is one JVM.

In [12]:
%%bash
cd /content/DistributedSystem
rm -rf output_local
hadoop jar hadoop-practice.jar -conf conf/hadoop-local.xml ./input ./output_local 2>&1 | tail
-n 8
echo "----- RESULT -----"
cat output_local/*

		BAD_ID=0
		CONNECTION=0
		IO_ERROR=0
		WRONG_LENGTH=0
		WRONG_MAP=0
		WRONG_REDUCE=0
	File Input Format Counters 
		Bytes Read=529
	File Output Format Counters 
		Bytes Written=29
----- RESULT -----
1949	111
1950	22


bash: line 4: -n: command not found


❓ **Question:** Is the result right? (Check one year by hand in the sample.) Does this run appear in YARN?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 4.3 Run on the cluster

Now the same jar on your pseudo-cluster, with the second configuration: copy `input/` to HDFS (`/input`), run the job with `-conf conf/hadoop-localhost.xml` reading `/input` and writing `/output`, and print the result **from HDFS**.

⏱️ On Colab's 2 vCPUs YARN takes a minute or two.

<details><summary>💡 Hint</summary>

`hadoop fs -rm -r -f /input /output`, `hadoop fs -mkdir /input`, `hadoop fs -copyFromLocal`, then `hadoop fs -cat /output/*`.

</details>

✅ **Expected:** The same two lines. Now input and output are HDFS paths and the job is an application run by YARN.

In [13]:
%%bash
cd /content/DistributedSystem
hadoop fs -rm -r -f /input /output
hadoop fs -mkdir /input
hadoop fs -copyFromLocal input/* /input
hadoop jar hadoop-practice.jar -conf conf/hadoop-localhost.xml /input /output 2>&1 | tail -n
8
echo "----- RESULT -----"
hadoop fs -cat /output/*

----- RESULT -----
1949	111
1950	22


tail: option requires an argument -- 'n'
Try 'tail --help' for more information.
bash: line 6: 8: command not found


❓ **Question:** Same result as before? What is different in the paths and in who executes the job?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 4.4 Override a property with `-D`

Run it again on the cluster but ask for **2 reducers** with `-D mapreduce.job.reduces=2` (the output goes to `/output2`). Then list the output folder in HDFS and print each part file.

<details><summary>💡 Hint</summary>

The generic options go **before** the program arguments: `hadoop jar <jar> -conf ... -D ... <in> <out>`. `hadoop fs -ls /output2` and `hadoop fs -cat /output2/part-r-00000`.

</details>

✅ **Expected:** Two part files (`part-r-00000`, `part-r-00001`). Each year goes to **exactly one** of them (the partitioner sends a key to a single reducer); with only 2 years it is quite possible that **both land in the same file** and the other one is empty. The options are parsed first by `ToolRunner`, so they must precede the input path.

In [14]:
%%bash
cd /content/DistributedSystem
hadoop fs -rm -r -f /output2
hadoop jar hadoop-practice.jar -conf conf/hadoop-localhost.xml -D mapreduce.job.reduces=2
/input /output2 2>&1 | tail -n 4
hadoop fs -ls /output2
for f in 0 1; do echo "--- part-r-0000$f"; hadoop fs -cat /output2/part-r-0000$f; done

bash: line 4: /input: No such file or directory
--- part-r-00000
--- part-r-00001


Usage: MaxTemperatureDriver [generic options] <input> <output>
Generic options supported are:
-conf <configuration file>        specify an application configuration file
-D <property=value>               define a value for a given property
-fs <file:///|hdfs://namenode:port> specify default filesystem URL to use, overrides 'fs.defaultFS' property from configurations.
-jt <local|resourcemanager:port>  specify a ResourceManager
-files <file1,...>                specify a comma-separated list of files to be copied to the map reduce cluster
-libjars <jar1,...>               specify a comma-separated list of jar files to be included in the classpath
-archives <archive1,...>          specify a comma-separated list of archives to be unarchived on the compute machines

The general command line syntax is:
command [genericOptions] [commandOptions]

ls: `/output2': No such file or directory
cat: `/output2/part-r-00000': No such file or directory
cat: `/output2/part-r-00001': No such file or direc

CalledProcessError: Command 'b'cd /content/DistributedSystem\nhadoop fs -rm -r -f /output2\nhadoop jar hadoop-practice.jar -conf conf/hadoop-localhost.xml -D mapreduce.job.reduces=2\n/input /output2 2>&1 | tail -n 4\nhadoop fs -ls /output2\nfor f in 0 1; do echo "--- part-r-0000$f"; hadoop fs -cat /output2/part-r-0000$f; done\n'' returned non-zero exit status 1.

❓ **Question:** How many files `part-r-0000x` do you get, and what is in each one? Why does the order of the generic options (`-conf`, `-D`) matter: before or after the input path?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

# Part 5 – 🧩 Final exercise: the **average** temperature per year

Our program computes the **maximum**. Your mission: change it to compute the **average temperature per year**, and run it on **your own data file**
(a synthetic NCDC file of ~700 records, different for every student, in the teacher's GitHub repository).

## 5.1 Download your data file

Your file is called `ncdc_<CODE>.txt`, where **CODE = the first 2 + the last 4 characters of your ID number**, in capital letters (the same code as your Sudoku). For example, the ID `12345678Z` gives `12678Z`.

`https://raw.githubusercontent.com/AntSanVar/DistributedSystemData/main/Session02/ncdc_<CODE>.txt`

Download it into `/content/DistributedSystem/mydata/` and check how many lines (records) it has and which years it covers (characters 16-19 of each line).

*(Alternative: download it on your computer and upload it with the **Files** panel 📁.)*

<details><summary>💡 Hint</summary>

Put your code in a shell variable. `mkdir -p mydata`, `wget -q -O mydata/ncdc.txt <url>`. A *404* error means a wrong code. To list the years: `cut -c16-19 mydata/ncdc.txt | sort | uniq -c`.

</details>

✅ **Expected:** About 700 lines and 8 consecutive years with 70-120 records each.

In [ ]:
%%bash
CODE=01476M # <- write YOUR code here
cd /content/DistributedSystem
mkdir -p mydata
wget -q -O mydata/ncdc.txt
https://raw.githubusercontent.com/AntSanVar/DistributedSystemData/main/Session02/ncd
c_$CODE.txt
wc -l mydata/ncdc.txt
cut -c16-19 mydata/ncdc.txt | sort | uniq -c

## 5.2 Baseline: the maximum with your data

First run the existing program (**maximum**) on your file, on the cluster, so you have something to compare with. Input folder in HDFS: `/mydata`; output: `/max_mine`.

<details><summary>💡 Hint</summary>

As in 4.3, but copy `mydata/ncdc.txt` to `/mydata`.

</details>

✅ **Expected:** 8 lines `year<TAB>maximum` (in tenths of °C).

In [ ]:
%%bash
cd /content/DistributedSystem
hadoop fs -rm -r -f /mydata /max_mine
hadoop fs -mkdir /mydata
hadoop fs -copyFromLocal mydata/ncdc.txt /mydata
hadoop jar hadoop-practice.jar -conf conf/hadoop-localhost.xml /mydata /max_mine 2>&1 |
tail -n 3
hadoop fs -cat /max_mine/*

## 5.3 Write the new Reducer

Create `src/main/java/AverageTemperatureReducer.java` (with `%%writefile`): for each year it receives all the valid temperatures and writes **the average in °C** (the data is in tenths of a degree, so divide by 10). The output value must be a `DoubleWritable`.

The Mapper does **not** change (it already filters the invalid readings).

<details><summary>💡 Hint</summary>

Same structure as `MaxTemperatureReducer`, but extends `Reducer<Text, IntWritable, Text, DoubleWritable>` (import `org.apache.hadoop.io.DoubleWritable`).
Loop over the values accumulating a `long sum` and an `int count`; then `context.write(key, new DoubleWritable(sum / (double) count / 10.0))`.

</details>

✅ **Expected:** “Writing ... AverageTemperatureReducer.java”.

In [ ]:
%%writefile /content/DistributedSystem/src/main/java/AverageTemperatureReducer.java
import java.io.IOException;
import org.apache.hadoop.io.DoubleWritable;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;
public class AverageTemperatureReducer
 extends Reducer<Text, IntWritable, Text, DoubleWritable> {
 @Override
 public void reduce(Text key, Iterable<IntWritable> values, Context context)
 throws IOException, InterruptedException {
 long sum = 0;
 int count = 0;
 for (IntWritable value : values) {
 sum += value.get();
 count++;
 }
 context.write(key, new DoubleWritable(sum / (double) count / 10.0));
 }
}

❓ **Question:** Can we reuse the Reducer as a Combiner now, as we did with the maximum? Why? (Think about what the average of two partial averages gives.)

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 5.4 Write the new Driver

Create `AverageTemperatureDriver.java`: a copy of `MaxTemperatureDriver` that uses **your** Reducer. Remember:

- **no Combiner** (see your answer above);
- the Mapper emits `(Text, IntWritable)` but the Reducer emits `(Text, DoubleWritable)`: the **map output types** must be declared with `setMapOutputKeyClass` / `setMapOutputValueClass`, and the **final output types** with `setOutputKeyClass` / `setOutputValueClass`;
- use `Job.getInstance(getConf(), "Average temperature")` (the `new Job(...)` constructor is deprecated).

<details><summary>💡 Hint</summary>

Start from `cp MaxTemperatureDriver.java AverageTemperatureDriver.java` and edit, or write it from scratch with `%%writefile`. The class name, the `Usage` message and `ToolRunner.run(new AverageTemperatureDriver(), args)` must also change.

</details>

✅ **Expected:** “Writing ... AverageTemperatureDriver.java”.

In [ ]:
%%writefile /content/DistributedSystem/src/main/java/AverageTemperatureDriver.java
import org.apache.hadoop.conf.Configured;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.DoubleWritable;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;
import org.apache.hadoop.util.Tool;
import org.apache.hadoop.util.ToolRunner;
public class AverageTemperatureDriver extends Configured implements Tool {
 @Override
 public int run(String[] args) throws Exception {
 if (args.length != 2) {
 System.err.printf("Usage: %s [generic options] <input> <output>\n",
getClass().getSimpleName());
 ToolRunner.printGenericCommandUsage(System.err);
 return -1;
 }
 Job job = Job.getInstance(getConf(), "Average temperature");
 job.setJarByClass(getClass());
 FileInputFormat.addInputPath(job, new Path(args[0]));
 FileOutputFormat.setOutputPath(job, new Path(args[1]));
 job.setMapperClass(MaxTemperatureMapper.class);
 job.setReducerClass(AverageTemperatureReducer.class);
 job.setMapOutputKeyClass(Text.class);
 job.setMapOutputValueClass(IntWritable.class);
 job.setOutputKeyClass(Text.class);
 job.setOutputValueClass(DoubleWritable.class);
 return job.waitForCompletion(true) ? 0 : 1;
 }
 public static void main(String[] args) throws Exception {
 System.exit(ToolRunner.run(new AverageTemperatureDriver(), args));
 }
}

## 5.5 Build a second JAR

Rebuild the project, but this time package a **new jar** called `hadoop-average.jar` whose main class is your Driver. You do not need to touch the POM: override the two properties it uses (`hadoop.mainClass` and `hadoop.fileName`) on the command line, besides the Hadoop version.

<details><summary>💡 Hint</summary>

`mvn -q -Dhadoop.version=$HADOOP_VERSION -Dhadoop.mainClass=... -Dhadoop.fileName=... package -DskipTests`. If it does not compile, read the first `ERROR` lines: they give the file and line.

</details>

✅ **Expected:** `hadoop-average.jar` (and the old `hadoop-practice.jar`).

In [ ]:
%%bash
cd /content/DistributedSystem
mvn -q -Dhadoop.version=$HADOOP_VERSION -
Dhadoop.mainClass=AverageTemperatureDriver -Dhadoop.fileName=hadoop-average
package -DskipTests
ls -l *.jar

## 5.6 Run it on your data

Run `hadoop-average.jar` **on the cluster** over your file (`/mydata`), writing to `/avg_mine`, and print the result.

<details><summary>💡 Hint</summary>

Same as 5.2 with the new jar and output folder.

</details>

✅ **Expected:** 8 lines `year<TAB>average` in °C, each average below the maximum of the same year.

In [ ]:
%%bash
cd /content/DistributedSystem
hadoop fs -rm -r -f /avg_mine
hadoop jar hadoop-average.jar -conf conf/hadoop-localhost.xml /mydata /avg_mine 2>&1 |
tail -n 3
echo "----- RESULT -----"
hadoop fs -cat /avg_mine/*

❓ **Question:** Is the average of each year plausible compared with the maxima of step 5.2?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

## 5.7 The same job in local mode (check)

Run the same jar in **local mode** over your local file (`mydata/`), writing to `avg_local`, and compare with 5.6.

<details><summary>💡 Hint</summary>

`-conf conf/hadoop-local.xml`, local input folder `mydata`.

</details>

✅ **Expected:** The same 8 lines.

In [ ]:
%%bash
cd /content/DistributedSystem
rm -rf avg_local
hadoop jar hadoop-average.jar -conf conf/hadoop-local.xml ./mydata ./avg_local 2>&1 | tail -
n 3
cat avg_local/*

❓ **Question:** Do both runs give exactly the same numbers? Why should they?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

### Your result

Paste the output of your average job (step 5.6) in the box below.

✍️ **Your result** *(double-click, then paste the 8 lines below, one per line inside this quote)*:

>

# Part 6 – Clean up

## 6.1 Stop the daemons

Stop YARN and HDFS and check with `jps` that nothing is left.

<details><summary>💡 Hint</summary>

Reverse order of start-up.

</details>

✅ **Expected:** Only `Jps` remains.

In [ ]:
%%bash
stop-yarn.sh
stop-dfs.sh
jps

### Final questions

Answer each one in the box below it.

❓ **Question:** Name the three Java classes of a MapReduce program and what each one does.

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

❓ **Question:** What is the Configuration API and what advantage does it give you when moving from your laptop to a cluster?

✍️ **Your answer** *(double-click this cell and write below this line)*:

>

❓ **Question:** In which case is it **not** valid to use the Reducer as a Combiner? Give an example other than the average.

✍️ **Your answer** *(double-click this cell and write below this line)*:

>